In [1]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup

import os
import numpy as np
from spikeinterface.core import concatenate_recordings

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import pandas as pd
import numpy as np
from matplotlib.collections import LineCollection
from probeinterface import write_probeinterface, read_probeinterface
import spikeinterface.exporters as sexp
from spikeinterface.core import write_binary_recording
from pathlib import Path
from typing import Dict, Iterable, List, Sequence, Tuple
from dataclasses import dataclass

import scipy
from scipy.sparse.csgraph import connected_components
import pickle

/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
@dataclass
class CliqueInfo:
    """Information about a clique (subset of channels)"""
    clique_id: int
    device_channel_indices: List[int]
    contact_ids: List[str]
    center: Tuple[float, float]

In [3]:
probe = read_probeinterface('/media/ubuntu/sda/mouse_test/probe/tip_probe_128_1.json')
if probe is None:
    raise ValueError("Recording does not have probe information")

# 定义按shank构建cliques的函数
def build_shank_cliques(probe, shank_boundaries=[250, 750, 1250]):
    """
    根据x坐标划分shank并构建cliques
    
    Parameters:
        probe: Probe对象
        shank_boundaries: shank之间的x坐标边界，默认[250, 750, 1250]
                         将probe划分为4个shank:
                         - shank 0: x < 250
                         - shank 1: 250 <= x < 750
                         - shank 2: 750 <= x < 1250
                         - shank 3: x >= 1250
    
    Returns:
        cliques: List[CliqueInfo] - 每个shank对应一个clique
    """
    from typing import List
    
    df = probe.to_dataframe()
    if "device_channel_indices" in df.columns:
        device_indices = df["device_channel_indices"].astype(int).to_numpy()
    else:
        device_indices = np.arange(len(df), dtype=int)
    positions = df.loc[:, ["x", "y"]].to_numpy()
    contact_ids = df["contact_ids"].astype(str).to_numpy()
    
    # 根据x坐标划分shank
    x_coords = positions[:, 0]
    shank_boundaries_sorted = sorted(shank_boundaries)
    
    cliques: List[CliqueInfo] = []
    
    # 定义shank范围
    shank_ranges = [
        (float('-inf'), shank_boundaries_sorted[0]),  # shank 0: x < 250
        (shank_boundaries_sorted[0], shank_boundaries_sorted[1]),  # shank 1: 250 <= x < 750
        (shank_boundaries_sorted[1], shank_boundaries_sorted[2]),  # shank 2: 750 <= x < 1250
        (shank_boundaries_sorted[2], float('inf')),  # shank 3: x >= 1250
    ]
    
    for shank_id, (x_min, x_max) in enumerate(shank_ranges):
        # 找到属于当前shank的通道
        if x_min == float('-inf'):
            mask = x_coords < x_max
        elif x_max == float('inf'):
            mask = x_coords >= x_min
        else:
            mask = (x_coords >= x_min) & (x_coords < x_max)
        
        shank_device_indices = device_indices[mask]
        shank_contact_ids = contact_ids[mask]
        shank_positions = positions[mask]
        
        if len(shank_device_indices) == 0:
            print(f"[WARNING] Shank {shank_id} has no channels")
            continue
        
        # 计算shank的中心位置
        center = tuple(np.mean(shank_positions, axis=0))
        
        # 创建CliqueInfo对象
        clique = CliqueInfo(
            clique_id=shank_id,
            device_channel_indices=list(shank_device_indices),
            contact_ids=list(shank_contact_ids),
            center=center,
        )
        cliques.append(clique)
        
        print(f"[INFO] Shank {shank_id}: {len(shank_device_indices)} channels "
              f"(x range: {x_min if x_min != float('-inf') else 'min'} to "
              f"{x_max if x_max != float('inf') else 'max'})")
    
    print(f"[INFO] Built {len(cliques)} cliques from {len(shank_boundaries) + 1} shanks")
    return cliques

# Build cliques from probe by shank
cliques = build_shank_cliques(probe, shank_boundaries=[250, 750, 1250])


[INFO] Shank 0: 32 channels (x range: min to 250)
[INFO] Shank 1: 32 channels (x range: 250 to 750)
[INFO] Shank 2: 32 channels (x range: 750 to 1250)
[INFO] Shank 3: 32 channels (x range: 1250 to max)
[INFO] Built 4 cliques from 4 shanks


In [4]:
file_dict = {
    'mouse2': {
        1214: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse&V1B_natima_251214_154409',
        1215: '/media/ubuntu/sda/mouse_test/raw_data/WLF_V1left&128ch2mouse_natima_251215_223556',
        1216: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse&V1left_natima_251216_214224',
        1217: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse2&V1left_natima_251217_220244',
        1218: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse2&V1od_natima_251218_214009',
        1219: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse2&V1left_natima_251219_192148'
    }

}

In [5]:
channel_list_A = ['A-000', 'A-001', 'A-002',
       'A-003', 'A-004', 'A-005', 'A-006', 'A-007', 'A-008', 'A-009',
       'A-010', 'A-011', 'A-012', 'A-013', 'A-014', 'A-015', 'A-016',
       'A-017', 'A-018', 'A-019', 'A-020', 'A-021', 'A-022', 'A-023',
       'A-024', 'A-025', 'A-026', 'A-027', 'A-028', 'A-029', 'A-030',
       'A-031', 'A-032', 'A-033', 'A-034', 'A-035', 'A-036', 'A-037',
       'A-038', 'A-039', 'A-040', 'A-041', 'A-042', 'A-043', 'A-044',
       'A-045', 'A-046', 'A-047', 'A-048', 'A-049', 'A-050', 'A-051',
       'A-052', 'A-053', 'A-054', 'A-055', 'A-056', 'A-057', 'A-058',
       'A-059', 'A-060', 'A-061', 'A-062', 'A-063', 'A-064', 'A-065',
       'A-066', 'A-067', 'A-068', 'A-069', 'A-070', 'A-071', 'A-072',
       'A-073', 'A-074', 'A-075', 'A-076', 'A-077', 'A-078', 'A-079',
       'A-080', 'A-081', 'A-082', 'A-083', 'A-084', 'A-085', 'A-086',
       'A-087', 'A-088', 'A-089', 'A-090', 'A-091', 'A-092', 'A-093',
       'A-094', 'A-095', 'A-096', 'A-097', 'A-098', 'A-099', 'A-100',
       'A-101', 'A-102', 'A-103', 'A-104', 'A-105', 'A-106', 'A-107',
       'A-108', 'A-109', 'A-110', 'A-111', 'A-112', 'A-113', 'A-114',
       'A-115', 'A-116', 'A-117', 'A-118', 'A-119', 'A-120', 'A-121',
       'A-122', 'A-123', 'A-124', 'A-125', 'A-126', 'A-127']

channel_list_B = ['B-000', 'B-001', 'B-002',
       'B-003', 'B-004', 'B-005', 'B-006', 'B-007', 'B-008', 'B-009',
       'B-010', 'B-011', 'B-012', 'B-013', 'B-014', 'B-015', 'B-016',
       'B-017', 'B-018', 'B-019', 'B-020', 'B-021', 'B-022', 'B-023',
       'B-024', 'B-025', 'B-026', 'B-027', 'B-028', 'B-029', 'B-030',
       'B-031', 'B-032', 'B-033', 'B-034', 'B-035', 'B-036', 'B-037',
       'B-038', 'B-039', 'B-040', 'B-041', 'B-042', 'B-043', 'B-044',
       'B-045', 'B-046', 'B-047', 'B-048', 'B-049', 'B-050', 'B-051',
       'B-052', 'B-053', 'B-054', 'B-055', 'B-056', 'B-057', 'B-058',
       'B-059', 'B-060', 'B-061', 'B-062', 'B-063', 'B-064', 'B-065',
       'B-066', 'B-067', 'B-068', 'B-069', 'B-070', 'B-071', 'B-072',
       'B-073', 'B-074', 'B-075', 'B-076', 'B-077', 'B-078', 'B-079',
       'B-080', 'B-081', 'B-082', 'B-083', 'B-084', 'B-085', 'B-086',
       'B-087', 'B-088', 'B-089', 'B-090', 'B-091', 'B-092', 'B-093',
       'B-094', 'B-095', 'B-096', 'B-097', 'B-098', 'B-099', 'B-100',
       'B-101', 'B-102', 'B-103', 'B-104', 'B-105', 'B-106', 'B-107',
       'B-108', 'B-109', 'B-110', 'B-111', 'B-112', 'B-113', 'B-114',
       'B-115', 'B-116', 'B-117', 'B-118', 'B-119', 'B-120', 'B-121',
       'B-122', 'B-123', 'B-124', 'B-125', 'B-126', 'B-127']

In [ ]:
for mouse_name, dates_dict in file_dict.items():
    print(f"\n{'='*60}")
    print(f"处理: {mouse_name}, 合并所有日期数据")
    print(f"{'='*60}")
    
    # 第一步：合并该mouse所有date的recording
    all_recordings_list = []
    channel_list = None
    
    for date, data_path in dates_dict.items():
        print(f"\n读取日期: {date}, 路径: {data_path}")
        
        # 获取该数据路径下的所有rhd文件
        file_list_path = Path(data_path)
        rhd_files = list(file_list_path.glob("*.rhd"))
        file_list = sorted(rhd_files)
        
        if len(file_list) == 0:
            print(f"警告: 在 {data_path} 中未找到.rhd文件，跳过")
            continue
        
        # 读取并合并该date的所有rhd文件
        recording_raw_list = []
        for file in file_list:
            recording_raw_list.append(se.read_intan(file, stream_id='0'))
        
        if len(recording_raw_list) > 0:
            date_recording = concatenate_recordings(recording_list=recording_raw_list)
            
            # 检测通道类型并选择对应的channel_list（只需要检测一次）
            
            available_channels = date_recording.get_channel_ids()
            if 'A-127' in available_channels:
                channel_list = channel_list_A
                print(f"检测到A通道，使用channel_list_A")
            elif 'B-127' in available_channels:
                channel_list = channel_list_B
                print(f"检测到B通道，使用channel_list_B")
            else:
                print(f"警告: 未找到A-127或B-127通道，可用通道: {available_channels[:10]}...")
                print(f"跳过此mouse")
                break
            
            # 选择通道
            date_recording = date_recording.select_channels(channel_list)
            
            # 统一将B开头的channel重命名为A开头（在选择通道之后）
            channel_ids = date_recording.get_channel_ids()
            new_channel_ids = []
            renamed_count = 0
            for ch_id in channel_ids:
                if isinstance(ch_id, str) and ch_id.startswith('B-'):
                    # 将B-000转换为A-000
                    new_ch_id = 'A-' + ch_id[2:]  # 保留'B-'之后的部分
                    new_channel_ids.append(new_ch_id)
                    renamed_count += 1
                else:
                    new_channel_ids.append(ch_id)
            
            if renamed_count > 0:
                # rename_channels需要传入完整的新channel IDs列表
                date_recording = date_recording.rename_channels(new_channel_ids)
                print(f"已将 {renamed_count} 个B开头channel重命名为A开头")
            
            all_recordings_list.append(date_recording)
            print(f"已添加日期 {date} 的数据，时长: {date_recording.get_total_duration():.2f}秒")
    
    if len(all_recordings_list) == 0:
        print(f"警告: {mouse_name} 没有有效数据，跳过")
        continue
    
    # 合并所有date的recording
    print(f"\n合并 {len(all_recordings_list)} 个日期的数据...")
    recording_combined = concatenate_recordings(recording_list=all_recordings_list)
    print(f"合并后总时长: {recording_combined.get_total_duration():.2f}秒")
    
    # 预处理合并后的数据
    print("\n开始预处理...")
    recording_raw = spre.unsigned_to_signed(recording_combined)
    recording_raw = spre.resample(recording_raw, 10000)
    recording_recorded = spre.bandpass_filter(recording_raw, freq_min=300, freq_max=3000)
    recording_recorded = spre.notch_filter(recording_recorded, freq=50)
    recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")

    probe = read_probeinterface('/media/ubuntu/sda/mouse_test/probe/tip_probe_128_1.json')
    recording_f = recording_f.set_probegroup(probe)

    for rep in range(5):    
        for clique in cliques:
            clique_id = clique.clique_id
            clique_channels = clique.contact_ids  # 使用contact_ids作为通道名
            
            print(f"\n{'='*60}")
            print(f"处理 {mouse_name} - Clique {clique_id} (通道数: {len(clique_channels)})")
            print(f"{'='*60}")
            
            # 选择该clique的通道
            # 先尝试使用contact_ids（字符串格式）
            try:
                recording_clique = recording_f.select_channels(clique_channels)
            except Exception as e1:
                # 如果失败，尝试使用device_channel_indices（整数索引）
                try:
                    # 获取recording的所有通道ID
                    all_channel_ids = recording_f.get_channel_ids()
                    # 使用device_channel_indices来选择通道
                    clique_channel_indices = clique.device_channel_indices
                    # 根据索引获取对应的通道ID
                    selected_channel_ids = [all_channel_ids[idx] for idx in clique_channel_indices if idx < len(all_channel_ids)]
                    recording_clique = recording_f.select_channels(selected_channel_ids)
                    print(f"使用device_channel_indices选择通道成功")
                except Exception as e2:
                    print(f"警告: 选择clique {clique_id} 的通道时出错")
                    print(f"尝试contact_ids失败: {e1}")
                    print(f"尝试device_channel_indices失败: {e2}")
                    print(f"contact_ids: {clique_channels[:5]}...")
                    print(f"device_channel_indices: {clique.device_channel_indices[:5]}...")
                    continue
            
            # 设置输出文件夹路径
            output_folder = f'/media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/{mouse_name}_{rep}/clique_{clique_id}'
            os.makedirs(output_folder, exist_ok=True)
            print(f"输出文件夹: {output_folder}")
            
            # 保存预处理后的数据
            recording_preprocessed = recording_clique.save(format="binary", n_jobs=20)

            default_params = {
                'detect_sign': 0,  
                'adjacency_radius': 120, 
                'freq_min': 300,  
                'freq_max': 3000,
                'filter': True,
                'whiten': True,  
                'num_workers': 30,
                'clip_size': 50,
                'detect_threshold': 5,
                'detect_interval': 3,  
            }
            sorting_mountainsort = ss.run_sorter(sorter_name='mountainsort4',
                                            recording=recording_preprocessed,
                                            remove_existing_folder='True',
                                            folder=output_folder,
                                            **default_params)


            analyzer_mountainsort = si.create_sorting_analyzer(
                sorting=sorting_mountainsort, 
                recording=recording_preprocessed, 
                format='binary_folder', 
                folder=output_folder + '/analyzer_kilosort4_binary'
            )

            # 计算扩展信息
            extensions_to_compute = [
                "random_spikes",
                "waveforms",
                "noise_levels",
                "templates",
                "unit_locations",
                "spike_locations",
                "correlograms",
                "template_similarity"
            ]

            extension_params = {
                "unit_locations": {"method": "center_of_mass"},
                "spike_locations": {"ms_before": 0.1},
                "correlograms": {"bin_ms": 0.1},
                "template_similarity": {"method": "cosine_similarity"}
            }

            analyzer_mountainsort.compute(extensions_to_compute, extension_params=extension_params, n_jobs=20)

            # 读取spikes.npy并检查无效的spike
            spikes_path = output_folder + "/analyzer_mountainsort4_binary/sorting/spikes.npy"
            if os.path.exists(spikes_path):
                spikes = np.load(spikes_path)
                
                # 获取recording的总样本数
                total_samples = recording_clique.get_num_samples()
                
                # 检查第一个和最后一个spike
                if len(spikes) > 0:
                    first_spike_valid = spikes[0]['sample_index'] >= 0
                    last_spike_valid = spikes[-1]['sample_index'] < total_samples
                    
                    # 如果第一个或最后一个spike无效，删除所有无效的spike
                    if not first_spike_valid or not last_spike_valid:
                        # 创建有效spike的掩码：sample_index >= 0 且 < total_samples
                        valid_mask = (spikes['sample_index'] >= 0) & (spikes['sample_index'] < total_samples)
                        spikes_filtered = spikes[valid_mask]
                        
                        # 保存过滤后的spikes
                        np.save(spikes_path, spikes_filtered)
                        print(f"删除了 {len(spikes) - len(spikes_filtered)} 个无效的spike")
                        print(f"原始spike数量: {len(spikes)}, 过滤后: {len(spikes_filtered)}")
                    else:
                        print("所有spike都在有效范围内")

            qm_params = sqm.get_default_qm_params()
            analyzer_mountainsort.compute("quality_metrics", qm_params, n_jobs=20)

            # 导出到phy格式
            sexp.export_to_phy(analyzer_mountainsort, output_folder + "/phy_folder_for_kilosort", verbose=True, n_jobs=20)
            
            print(f"完成处理: {mouse_name} - Clique {clique_id}\n")

print("\n所有数据处理完成！")


In [6]:
def process_unified_sorting(
    recording_cmr,
    output_folder,
    distance_threshold: float = 10.0,
    similarity_threshold: float = 0.95,
    peak_sign: str= 'neg',
    n_jobs: int = 20,
    verbose: bool = False
):
    from spikeinterface.qualitymetrics import compute_quality_metrics

    sampling_frequency = recording_cmr.get_sampling_frequency()

    if verbose:
        print(f"\n{'='*60}")
        print(f"开始统一处理整个recording的sorting结果")
        print(f"{'='*60}\n")

    # 统一的phy_folder路径
    phy_folder = f'{output_folder}/phy_folder_for_kilosort'

    # 读取整个recording的sorting结果
    if verbose:
        print("读取统一的sorting结果...")
    sorting_curated_phy = se.read_phy(phy_folder)
    if verbose:
        print(f"读取到 {len(sorting_curated_phy.unit_ids)} 个units\n")

    # 创建analyzer并计算extensions
    if verbose:
        print("创建analyzer并计算extensions...")
    analyzer_curated_phy = si.create_sorting_analyzer(
        sorting=sorting_curated_phy,
        recording=recording_cmr,
        format='binary_folder',
        folder=output_folder + '/analyzer_curated_temp_raw',
        n_jobs=n_jobs,
        verbose=verbose
    )

    extensions_to_compute = [
        "random_spikes",
        "waveforms",
        "templates",
        "unit_locations",
        "template_similarity"
    ]

    extension_params = {
        "unit_locations": {"method": "center_of_mass"},
        "template_similarity": {"method": "cosine_similarity"}
    }

    analyzer_curated_phy.compute(
        extensions_to_compute,
        extension_params=extension_params,
        n_jobs=n_jobs,
        verbose=verbose
    )
    if verbose:
        print("完成extensions计算\n")

    # 获取neuron信息
    templates_ext = analyzer_curated_phy.get_extension("templates")
    templates_dense = templates_ext.data["average"]
    sparsity = analyzer_curated_phy.sparsity
    unit_locations_ext = analyzer_curated_phy.get_extension("unit_locations")
    unit_locations = unit_locations_ext.get_data()
    channel_locations = analyzer_curated_phy.get_channel_locations()

    # 处理merge逻辑
    if unit_locations.shape[1] >= 2:
        unit_distances = scipy.spatial.distance.cdist(
            unit_locations[:, :2],
            unit_locations[:, :2],
            metric="euclidean"
        )
    else:
        unit_distances = scipy.spatial.distance.cdist(
            unit_locations,
            unit_locations,
            metric="euclidean"
        )

    template_similarity_ext = analyzer_curated_phy.get_extension("template_similarity")
    template_similarity = template_similarity_ext.get_data()

    num_units = len(analyzer_curated_phy.unit_ids)
    pair_mask = np.zeros((num_units, num_units), dtype=bool)

    for i in range(num_units):
        for j in range(i + 1, num_units):
            if unit_distances[i, j] < distance_threshold and template_similarity[i, j] > similarity_threshold:
                pair_mask[i, j] = True
                pair_mask[j, i] = True

    n_components, labels = connected_components(
        csgraph=pair_mask,
        directed=False,
        return_labels=True
    )

    merge_unit_groups = []
    unit_ids_list = analyzer_curated_phy.unit_ids
    for component_id in range(n_components):
        unit_indices = np.where(labels == component_id)[0]
        if len(unit_indices) > 1:
            group = [unit_ids_list[i] for i in unit_indices]
            merge_unit_groups.append(group)

    if len(merge_unit_groups) > 0:
        print(f"发现 {len(merge_unit_groups)} 组需要merge的units，开始merge...")
        analyzer_merged = analyzer_curated_phy.merge_units(
            merge_unit_groups=merge_unit_groups,
            censor_ms=0.3,
            merging_mode="hard",
            new_id_strategy="append",
            format='binary_folder',
            folder=output_folder + '/analyzer_merged_raw',
            verbose=True,
            n_jobs=20
        )
        
        analyzer_merged.compute(extensions_to_compute, extension_params=extension_params, n_jobs=20, verbose = False)
        
        templates_ext_final = analyzer_merged.get_extension("templates")
        templates_dense_final = templates_ext_final.data["average"]
        sparsity_final = analyzer_merged.sparsity
        unit_locations_ext_final = analyzer_merged.get_extension("unit_locations")
        unit_locations_final = unit_locations_ext_final.get_data()
        channel_locations_final = analyzer_merged.get_channel_locations()
        sorting_final = analyzer_merged.sorting
        unit_ids_list_final = analyzer_merged.unit_ids
        
        # 生成position_waveforms
        position_waveforms_final = []
        for unit_id in unit_ids_list_final:
            unit_index = analyzer_merged.sorting.id_to_index(unit_id)
            template_dense_unit = templates_dense_final[unit_index, :, :]
            template_sparse_unit = sparsity_final.sparsify_waveforms(template_dense_unit[np.newaxis, :, :], unit_id)[0]
            sparse_channel_indices = sparsity_final.unit_id_to_channel_indices[unit_id]
            
            if len(sparse_channel_indices) == 0:
                position_waveform = np.zeros(templates_dense_final.shape[1], dtype=templates_dense_final.dtype)
                position_waveforms_final.append(position_waveform)
                continue
            
            sparse_channel_locations = channel_locations_final[sparse_channel_indices, :2]
            unit_location = unit_locations_final[unit_index, :2]
            
            distances = np.sqrt(np.sum((sparse_channel_locations - unit_location[np.newaxis, :])**2, axis=1))
            epsilon = 1e-10
            weights = 1.0 / (distances + epsilon)
            weights = weights / np.sum(weights)
            
            position_waveform = np.dot(template_sparse_unit, weights)
            position_waveforms_final.append(position_waveform)
        
        position_waveforms_final = np.array(position_waveforms_final)
        extremum_channels_final = si.get_template_extremum_channel(
            analyzer_merged, 
            peak_sign=peak_sign,
            outputs="id"
        )
        
        channel_ids_list = list(analyzer_merged.recording.get_channel_ids())
    else:
        print("无需merge units\n")
        # 不需要merge，使用原始结果
        templates_ext_final = analyzer_curated_phy.get_extension("templates")
        templates_dense_final = templates_ext_final.data["average"]
        sparsity_final = analyzer_curated_phy.sparsity
        unit_locations_ext_final = analyzer_curated_phy.get_extension("unit_locations")
        unit_locations_final = unit_locations_ext_final.get_data()
        channel_locations_final = analyzer_curated_phy.get_channel_locations()
        sorting_final = analyzer_curated_phy.sorting
        unit_ids_list_final = unit_ids_list
        
        # 生成position_waveforms
        position_waveforms_final = []
        for unit_id in unit_ids_list_final:
            unit_index = analyzer_curated_phy.sorting.id_to_index(unit_id)
            template_dense_unit = templates_dense_final[unit_index, :, :]
            template_sparse_unit = sparsity_final.sparsify_waveforms(template_dense_unit[np.newaxis, :, :], unit_id)[0]
            sparse_channel_indices = sparsity_final.unit_id_to_channel_indices[unit_id]
            
            if len(sparse_channel_indices) == 0:
                position_waveform = np.zeros(templates_dense_final.shape[1], dtype=templates_dense_final.dtype)
                position_waveforms_final.append(position_waveform)
                continue
            
            sparse_channel_locations = channel_locations_final[sparse_channel_indices, :2]
            unit_location = unit_locations_final[unit_index, :2]
            
            distances = np.sqrt(np.sum((sparse_channel_locations - unit_location[np.newaxis, :])**2, axis=1))
            epsilon = 1e-10
            weights = 1.0 / (distances + epsilon)
            weights = weights / np.sum(weights)
            
            position_waveform = np.dot(template_sparse_unit, weights)
            position_waveforms_final.append(position_waveform)
        
        position_waveforms_final = np.array(position_waveforms_final)
        extremum_channels_final = si.get_template_extremum_channel(
            analyzer_curated_phy, 
            peak_sign=peak_sign,
            outputs="id"
        )
        
        channel_ids_list = list(analyzer_curated_phy.recording.get_channel_ids())

    print("计算每个unit的channel_id...")
    channel_ids_dict = {}  # {unit_id: [contact_id1, contact_id2, ...]}
    for idx, unit_id in enumerate(unit_ids_list_final):
        unit_index = sorting_final.id_to_index(unit_id)
        template_unit = templates_dense_final[unit_index, :, :]  # (n_samples, n_channels)
        
        # 找到template中值不为0的通道
        non_zero_channels = []
        for ch_idx in range(template_unit.shape[1]):  # 遍历channels（最后一个维度）
            if np.any(template_unit[:, ch_idx] != 0):  # 检查该通道在所有时间点的值
                # recording的channel_id已经是contact_id，直接使用
                contact_id = str(channel_ids_list[ch_idx])
                non_zero_channels.append(contact_id)
        
        channel_ids_dict[unit_id] = non_zero_channels

    print(f"完成channel_id计算，共处理{len(channel_ids_dict)}个units\n")

    # 计算channel_snr（每个unit的各个channel的SNR）
    print("计算channel_snr...")
    duration_samples = int(10 * sampling_frequency)  # 10秒
    max_samples = min(duration_samples, recording_cmr.get_num_samples())
    traces = recording_cmr.get_traces(start_frame=0, end_frame=max_samples)  # (n_timepoints, n_channels)

    noise_std_detect = np.median(np.abs(traces) / 0.6745, axis=0)  # (n_channels,)

    all_spike_times = []
    all_spike_unit_ids = []
    for unit_id in unit_ids_list_final:
        spike_train = sorting_final.get_unit_spike_train(unit_id)
        all_spike_times.extend(spike_train.tolist())
        all_spike_unit_ids.extend([unit_id] * len(spike_train))

    n_spikes_total = len(all_spike_times)
    n_spikes_sample = min(1000, n_spikes_total)
    if n_spikes_sample > 0:
        random_indices = np.random.choice(n_spikes_total, size=n_spikes_sample, replace=False)
        sampled_spike_times = [all_spike_times[i] for i in random_indices]
        sampled_spike_unit_ids = [all_spike_unit_ids[i] for i in random_indices]
    else:
        sampled_spike_times = []
        sampled_spike_unit_ids = []

    left_sample = 10
    right_sample = 20
    window_size = left_sample + right_sample

    channel_snr_dict = {} 

    for unit_id in unit_ids_list_final:
        channel_snr_dict[unit_id] = {}
        unit_spike_times = [st for st, uid in zip(sampled_spike_times, sampled_spike_unit_ids) if uid == unit_id]
        
        if len(unit_spike_times) == 0:
            unit_spike_times = sorting_final.get_unit_spike_train(unit_id).tolist()
            if len(unit_spike_times) > 1000:
                unit_spike_times = np.random.choice(unit_spike_times, size=1000, replace=False).tolist()
        
        unit_waveforms = []  # List of (n_channels, window_size)
        valid_spike_times = []
        
        for spike_time in unit_spike_times:
            start = spike_time - left_sample
            end = spike_time + right_sample

            if start < 0:
                start = 0
            if end > recording_cmr.get_num_samples():
                end = recording_cmr.get_num_samples()
            
            waveform = recording_cmr.get_traces(start_frame=start, end_frame=end)  # (n_timepoints, n_channels)
            unit_waveforms.append(waveform)
            valid_spike_times.append(spike_time)
        
        if len(unit_waveforms) == 0:
            continue
        
        unit_waveforms = np.array(unit_waveforms)  # (n_spikes, n_timepoints, n_channels)
        
        spike_time_values = unit_waveforms[:, left_sample, :]  # (n_spikes, n_channels) - 每个spike在spike_time时刻各个channel的值
        
        channel_amplitudes = np.mean(spike_time_values, axis=0)  # (n_channels,) - 每个channel的平均值（在spike_time时刻）
        channel_snr = np.abs(channel_amplitudes) / noise_std_detect  # (n_channels,)
        
        unit_channel_ids = channel_ids_dict.get(unit_id, [])  # 获取该unit的channel_id列表
        
        for ch_idx, snr_value in enumerate(channel_snr):
            channel_id = str(channel_ids_list[ch_idx])
            if channel_id in unit_channel_ids:
                channel_snr_dict[unit_id][channel_id] = float(snr_value)

    print(f"完成channel_snr计算，共处理{len(channel_snr_dict)}个units\n")

    # 创建channel_id到索引的映射
    channel_id_to_idx = {str(ch_id): idx for idx, ch_id in enumerate(channel_ids_list)}

    # 计算每个unit的sign（根据extremum_channel处template的极值）
    unit_signs = {}
    for idx, unit_id in enumerate(unit_ids_list_final):
        extremum_channel = extremum_channels_final[unit_id]
        extremum_channel_str = str(extremum_channel)

        # 获取extremum_channel对应的索引
        if extremum_channel_str in channel_id_to_idx:
            ch_idx = channel_id_to_idx[extremum_channel_str]
            template_unit = templates_dense_final[idx, :, ch_idx]  # (n_samples,)

            # 计算template在该通道的极值
            template_min = np.min(template_unit)
            template_max = np.max(template_unit)

            # 判断极值符号：比较绝对值
            if np.abs(template_max) >= np.abs(template_min):
                unit_signs[unit_id] = 1  # 正极值
            else:
                unit_signs[unit_id] = -1  # 负极值
        else:
            # 如果找不到对应的channel，默认设为-1
            unit_signs[unit_id] = -1

    neuron_inf_all = {}
    for idx, unit_id in enumerate(unit_ids_list_final):
        neuron_inf_all[unit_id] = {
            'location_x': float(unit_locations_final[idx, 0]),
            'location_y': float(unit_locations_final[idx, 1]),
            'position_waveform': position_waveforms_final[idx],
            'extremum_channel': extremum_channels_final[unit_id],
            'sign': unit_signs[unit_id],
            'channel_id': channel_ids_dict[unit_id],
            'channel_snr': channel_snr_dict.get(unit_id, {})
        }

    print("生成整体的gt_detect_array...")
    spike_vector_final = sorting_final.to_spike_vector()
    gt_detect_data_all = []

    for spike in spike_vector_final:
        unit_index = spike['unit_index']
        unit_id = sorting_final.unit_ids[unit_index]        
        sample_index = spike['sample_index']

        extremum_channel = extremum_channels_final[unit_id]
        
        gt_detect_data_all.append({
            'time': sample_index,
            'unit_id': unit_id,
            'extremum_channel': str(extremum_channel),
        })

    gt_detect_array_all = pd.DataFrame(gt_detect_data_all)

    if verbose:
        print("保存neuron_inf_all和gt_detect_array_all...")
    with open(output_folder + '/neuron_inf_raw.pickle', 'wb') as f:
        pickle.dump(neuron_inf_all, f)
    gt_detect_array_all.to_csv(output_folder + '/gt_detect_array_raw.csv', index=False)
    if verbose:
        print(f"已保存到: {output_folder}/neuron_inf_all.pickle 和 {output_folder}/gt_detect_array_all.csv\n")

    if len(merge_unit_groups) > 0:
        analyzer_final = analyzer_merged
    else:
        analyzer_final = analyzer_curated_phy

    return neuron_inf_all, gt_detect_array_all, analyzer_final

In [7]:
for mouse_name, dates_dict in file_dict.items():
    print(f"\n{'='*60}")
    print(f"处理: {mouse_name}, 合并所有日期数据")
    print(f"{'='*60}")
    
    # 第一步：合并该mouse所有date的recording
    all_recordings_list = []
    channel_list = None
    
    for date, data_path in dates_dict.items():
        print(f"\n读取日期: {date}, 路径: {data_path}")
        
        # 获取该数据路径下的所有rhd文件
        file_list_path = Path(data_path)
        rhd_files = list(file_list_path.glob("*.rhd"))
        file_list = sorted(rhd_files)
        
        if len(file_list) == 0:
            print(f"警告: 在 {data_path} 中未找到.rhd文件，跳过")
            continue
        
        # 读取并合并该date的所有rhd文件
        recording_raw_list = []
        for file in file_list:
            recording_raw_list.append(se.read_intan(file, stream_id='0'))
        
        if len(recording_raw_list) > 0:
            date_recording = concatenate_recordings(recording_list=recording_raw_list)
            
            # 检测通道类型并选择对应的channel_list（只需要检测一次）
            
            available_channels = date_recording.get_channel_ids()
            if 'A-127' in available_channels:
                channel_list = channel_list_A
                print(f"检测到A通道，使用channel_list_A")
            elif 'B-127' in available_channels:
                channel_list = channel_list_B
                print(f"检测到B通道，使用channel_list_B")
            else:
                print(f"警告: 未找到A-127或B-127通道，可用通道: {available_channels[:10]}...")
                print(f"跳过此mouse")
                break
            
            # 选择通道
            date_recording = date_recording.select_channels(channel_list)
            
            # 统一将B开头的channel重命名为A开头（在选择通道之后）
            channel_ids = date_recording.get_channel_ids()
            new_channel_ids = []
            renamed_count = 0
            for ch_id in channel_ids:
                if isinstance(ch_id, str) and ch_id.startswith('B-'):
                    # 将B-000转换为A-000
                    new_ch_id = 'A-' + ch_id[2:]  # 保留'B-'之后的部分
                    new_channel_ids.append(new_ch_id)
                    renamed_count += 1
                else:
                    new_channel_ids.append(ch_id)
            
            if renamed_count > 0:
                # rename_channels需要传入完整的新channel IDs列表
                date_recording = date_recording.rename_channels(new_channel_ids)
                print(f"已将 {renamed_count} 个B开头channel重命名为A开头")
            
            all_recordings_list.append(date_recording)
            print(f"已添加日期 {date} 的数据，时长: {date_recording.get_total_duration():.2f}秒")
    
    if len(all_recordings_list) == 0:
        print(f"警告: {mouse_name} 没有有效数据，跳过")
        continue
    
    # 合并所有date的recording
    print(f"\n合并 {len(all_recordings_list)} 个日期的数据...")
    recording_combined = concatenate_recordings(recording_list=all_recordings_list)
    print(f"合并后总时长: {recording_combined.get_total_duration():.2f}秒")
    
    # 预处理合并后的数据
    print("\n开始预处理...")
    recording_raw = spre.unsigned_to_signed(recording_combined)
    recording_raw = spre.resample(recording_raw, 10000)
    recording_recorded = spre.bandpass_filter(recording_raw, freq_min=300, freq_max=3000)
    recording_recorded = spre.notch_filter(recording_recorded, freq=50)
    recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")

    probe = read_probeinterface('/media/ubuntu/sda/mouse_test/probe/tip_probe_128_1.json')
    recording_f = recording_f.set_probegroup(probe)

    for rep in range(5):    
        for clique in cliques:
            clique_id = clique.clique_id
            clique_channels = clique.contact_ids  # 使用contact_ids作为通道名
            
            print(f"\n{'='*60}")
            print(f"处理 {mouse_name} - Clique {clique_id} (通道数: {len(clique_channels)})")
            print(f"{'='*60}")
            
            try:
                recording_clique = recording_f.select_channels(clique_channels)
            except Exception as e1:
                # 如果失败，尝试使用device_channel_indices（整数索引）
                try:
                    # 获取recording的所有通道ID
                    all_channel_ids = recording_f.get_channel_ids()
                    # 使用device_channel_indices来选择通道
                    clique_channel_indices = clique.device_channel_indices
                    # 根据索引获取对应的通道ID
                    selected_channel_ids = [all_channel_ids[idx] for idx in clique_channel_indices if idx < len(all_channel_ids)]
                    recording_clique = recording_f.select_channels(selected_channel_ids)
                    print(f"使用device_channel_indices选择通道成功")
                except Exception as e2:
                    print(f"警告: 选择clique {clique_id} 的通道时出错")
                    print(f"尝试contact_ids失败: {e1}")
                    print(f"尝试device_channel_indices失败: {e2}")
                    print(f"contact_ids: {clique_channels[:5]}...")
                    print(f"device_channel_indices: {clique.device_channel_indices[:5]}...")
                    continue
            
            output_folder = f'/media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/{mouse_name}_{rep}/clique_{clique_id}'
            os.makedirs(output_folder, exist_ok=True)
            print(f"输出文件夹: {output_folder}")
            
            recording_preprocessed = recording_clique.save(format="binary", n_jobs=20)
            neuron_inf_all, gt_detect_array_all, analyzer = process_unified_sorting(
                    recording_cmr=recording_preprocessed,
                    output_folder=output_folder ,
                    distance_threshold=10.0,
                    similarity_threshold=0.95,
                    peak_sign='both',
                    n_jobs=30,
                    verbose=True
                )
            print(f"完成处理: {mouse_name} - Clique {clique_id}\n")

print("\n所有数据处理完成！")



处理: mouse2, 合并所有日期数据

读取日期: 1214, 路径: /media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse&V1B_natima_251214_154409
检测到B通道，使用channel_list_B
已将 128 个B开头channel重命名为A开头
已添加日期 1214 的数据，时长: 1587.35秒

读取日期: 1215, 路径: /media/ubuntu/sda/mouse_test/raw_data/WLF_V1left&128ch2mouse_natima_251215_223556
检测到B通道，使用channel_list_B
已将 128 个B开头channel重命名为A开头
已添加日期 1215 的数据，时长: 1280.72秒

读取日期: 1216, 路径: /media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse&V1left_natima_251216_214224
检测到B通道，使用channel_list_B
已将 128 个B开头channel重命名为A开头
已添加日期 1216 的数据，时长: 1293.75秒

读取日期: 1217, 路径: /media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse2&V1left_natima_251217_220244
检测到A通道，使用channel_list_A
已添加日期 1217 的数据，时长: 2171.02秒

读取日期: 1218, 路径: /media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse2&V1od_natima_251218_214009
检测到A通道，使用channel_list_A
已添加日期 1218 的数据，时长: 2773.13秒

读取日期: 1219, 路径: /media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse2&V1left_natima_251219_192148
检测到A通道，使用channel_list_A
已添加日期 1219 的数据，时长: 1702.

write_binary_recording (workers: 20 processes): 100%|██████████| 10809/10809 [02:28<00:00, 72.79it/s]



开始统一处理整个recording的sorting结果

读取统一的sorting结果...
读取到 65 个units

创建analyzer并计算extensions...
estimate_sparsity 
engine=process - n_jobs=30 - samples_per_chunk=10,000 - chunk_memory=625.00 KiB - total_memory=18.31 MiB - chunk_duration=1.00s


estimate_sparsity (workers: 30 processes): 100%|██████████| 10809/10809 [00:01<00:00, 9025.63it/s]

compute_waveforms 
engine=process - n_jobs=30 - samples_per_chunk=10,000 - chunk_memory=625.00 KiB - total_memory=18.31 MiB - chunk_duration=1.00s


...


开始统一处理整个recording的sorting结果

读取统一的sorting结果...
读取到 67 个units

创建analyzer并计算extensions...
estimate_sparsity 
engine=process - n_jobs=30 - samples_per_chunk=10,000 - chunk_memory=625.00 KiB - total_memory=18.31 MiB - chunk_duration=1.00s


estimate_sparsity (workers: 30 processes): 100%|██████████| 10809/10809 [00:00<00:00, 11310.12it/s]


compute_waveforms 
engine=process - n_jobs=30 - samples_per_chunk=10,000 - chunk_memory=625.00 KiB - total_memory=18.31 MiB - chunk_duration=1.00s


compute_waveforms (workers: 30 processes): 100%|██████████| 10809/10809 [00:34<00:00, 313.65it/s]


完成extensions计算

无需merge units

计算每个unit的channel_id...
完成channel_id计算，共处理67个units

计算channel_snr...
完成channel_snr计算，共处理67个units

生成整体的gt_detect_array...
保存neuron_inf_all和gt_detect_array_all...
已保存到: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2_4/clique_3/neuron_inf_all.pickle 和 /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2_4/clique_3/gt_detect_array_all.csv

完成处理: mouse2 - Clique 3


所有数据处理完成！


In [8]:
# ==================== 5x5 重复间匹配准确率分析 ====================

import pickle
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from scipy.stats import pearsonr
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# 导入必要的函数
from utils_clique import map_gt_annotation, match_neurons

# 数据路径
base_path = '/media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort'

# ==================== 创建通道名称到索引的映射 ====================
def create_channel_name_to_index_map():
    """
    创建通道名称到整数索引的映射
    通道命名规则: A-000 到 A-127, B-000 到 B-127
    """
    channel_to_idx = {}
    # A-000 到 A-127 -> 索引 0-127
    for i in range(128):
        channel_name = f'A-{i:03d}'
        channel_to_idx[channel_name] = i
    # B-000 到 B-127 -> 索引 128-255
    for i in range(128):
        channel_name = f'B-{i:03d}'
        channel_to_idx[channel_name] = 128 + i
    return channel_to_idx

# 创建映射字典
CHANNEL_TO_IDX = create_channel_name_to_index_map()

def convert_channel_to_index(channel_name):
    """
    将通道名称转换为整数索引
    """
    if isinstance(channel_name, (int, np.integer)):
        return int(channel_name)
    return CHANNEL_TO_IDX.get(str(channel_name), -1)

# ==================== 1. 读取数据 ====================
print("=" * 60)
print("步骤1: 读取 mouse2_0 到 mouse2_4 的数据")
print("=" * 60)

neuron_inf_all_list = []
gt_detect_array_list = []

for rep_idx in range(5):
    mouse_path = f'{base_path}/mouse2_{rep_idx}/clique_1'
    
    # 读取 neuron_inf_all.pickle
    neuron_inf_path = f'{mouse_path}/neuron_inf_raw.pickle'
    with open(neuron_inf_path, 'rb') as f:
        neuron_inf_all = pickle.load(f)
    neuron_inf_all_list.append(neuron_inf_all)
    
    # 读取 gt_detect_array.csv
    gt_path = f'{mouse_path}/gt_detect_array_raw.csv'
    gt_detect = pd.read_csv(gt_path)
    gt_detect_array_list.append(gt_detect)
    
    print(f"Rep {rep_idx}: {len(neuron_inf_all)} neurons, {len(gt_detect)} spikes")

print(f"\n共加载 {len(neuron_inf_all_list)} 个重复的数据")

# ==================== 2. gt_detect_array 匹配准确率矩阵 ====================
print("\n" + "=" * 60)
print("步骤2: 计算 gt_detect_array 5x5 匹配准确率矩阵")
print("=" * 60)

def compute_gt_match_accuracy(detect_array, gt_array):
    """
    计算 gt 检测数组的匹配准确率
    """
    if len(detect_array) == 0 or len(gt_array) == 0:
        return 0.0
    
    # 转换为 numpy 数组，并将通道名称转换为整数索引
    detect_times = detect_array['time'].to_numpy().astype(np.int64)
    detect_channels = detect_array['extremum_channel'].apply(convert_channel_to_index).to_numpy().astype(np.int64)
    gt_times = gt_array['time'].to_numpy().astype(np.int64)
    gt_channels = gt_array['extremum_channel'].apply(convert_channel_to_index).to_numpy().astype(np.int64)
    
    # 组合成 numpy 数组
    detect_np = np.column_stack([detect_times, detect_channels])
    gt_np = np.column_stack([gt_times, gt_channels])
    
    # 使用 map_gt_annotation 进行匹配
    matched_gt_indices = map_gt_annotation(detect_np, gt_np)
    
    # 计算准确率 (匹配的 spike 数量 / gt_array 总 spike 数量)
    n_matched = np.sum(matched_gt_indices >= 0)
    accuracy = n_matched / len(gt_array)
    
    return accuracy

# 计算 5x5 矩阵
n_reps = 5
gt_accuracy_matrix = np.zeros((n_reps, n_reps))

for i in range(n_reps):
    for j in range(n_reps):
        accuracy = compute_gt_match_accuracy(gt_detect_array_list[i], gt_detect_array_list[j])
        gt_accuracy_matrix[i, j] = accuracy
        print(f"  Rep {i} vs Rep {j}: {accuracy:.4f}")

print("\ngt_detect_array 匹配准确率矩阵:")
print(gt_accuracy_matrix)

# 可视化 gt_detect_array 匹配矩阵
fig1, ax1 = plt.subplots(figsize=(8, 6))
sns.heatmap(gt_accuracy_matrix, annot=True, fmt='.3f', cmap='viridis', 
            xticklabels=[f'Rep {i}' for i in range(n_reps)],
            yticklabels=[f'Rep {i}' for i in range(n_reps)],
            ax=ax1, vmin=0, vmax=1)
ax1.set_title('gt_detect_array Matching Accuracy Matrix\n(Row: Detect, Column: GT)', fontsize=12)
ax1.set_xlabel('Ground Truth', fontsize=10)
ax1.set_ylabel('Detection', fontsize=10)

# ==================== 3. neuron_inf 匹配准确率矩阵 ====================
print("\n" + "=" * 60)
print("步骤3: 计算 neuron_inf 5x5 匹配准确率矩阵")
print("=" * 60)

def convert_neuron_inf_all_to_dataframe(neuron_inf_all):
    """
    将 neuron_inf_all 字典转换为 match_neurons 所需的 DataFrame 格式
    """
    data = []
    for unit_id, info in neuron_inf_all.items():
        data.append({
            'Neuron': str(unit_id),
            'position_1': info['location_x'],
            'position_2': info['location_y'],
            'position_waveform': info['position_waveform']
        })
    return pd.DataFrame(data)

# 转换所有 neuron_inf_all 为 DataFrame
neuron_inf_df_list = []
for i, neuron_inf_all in enumerate(neuron_inf_all_list):
    df = convert_neuron_inf_all_to_dataframe(neuron_inf_all)
    neuron_inf_df_list.append(df)
    print(f"Rep {i}: {len(df)} neurons converted to DataFrame")

# 计算 5x5 匹配矩阵
neuron_accuracy_matrix = np.zeros((n_reps, n_reps))

for i in range(n_reps):
    for j in range(n_reps):
        # 使用 match_neurons 进行匹配
        matched_result = match_neurons(
            train_neuron_inf=neuron_inf_df_list[i],
            eval_neuron_inf=neuron_inf_df_list[j],
            position_threshold=10,
            waveform_similarity_threshold=0.95
        )
        
        # 计算准确率 (匹配的 neuron 数量 / 评估集 neuron 总数)
        n_matched = np.sum(matched_result['neuron_match'] != 'unmatch')
        accuracy = n_matched / len(matched_result)
        neuron_accuracy_matrix[i, j] = accuracy
        
        print(f"  Rep {i} vs Rep {j}: {accuracy:.4f} ({n_matched}/{len(matched_result)})")

print("\nneuron_inf 匹配准确率矩阵:")
print(neuron_accuracy_matrix)

# 可视化 neuron_inf 匹配矩阵
fig2, ax2 = plt.subplots(figsize=(8, 6))
sns.heatmap(neuron_accuracy_matrix, annot=True, fmt='.3f', cmap='viridis',
            xticklabels=[f'Rep {i}' for i in range(n_reps)],
            yticklabels=[f'Rep {i}' for i in range(n_reps)],
            ax=ax2, vmin=0, vmax=1)
ax2.set_title('neuron_inf Matching Accuracy Matrix\n(Row: Train, Column: Eval, Threshold: Pos=10, WF=0.95)', fontsize=12)
ax2.set_xlabel('Evaluation Set', fontsize=10)
ax2.set_ylabel('Training Set', fontsize=10)

# ==================== 4. 保存结果到 PDF ====================
print("\n" + "=" * 60)
print("步骤4: 保存结果")
print("=" * 60)

output_pdf = f'{base_path}/matching_accuracy_analysis.pdf'

with PdfPages(output_pdf) as pdf:
    # 保存 gt_detect_array 矩阵图
    fig1.tight_layout()
    pdf.savefig(fig1, bbox_inches='tight')
    plt.close(fig1)
    
    # 保存 neuron_inf 矩阵图
    fig2.tight_layout()
    pdf.savefig(fig2, bbox_inches='tight')
    plt.close(fig2)
    
    # 保存矩阵数据到 CSV
    gt_df = pd.DataFrame(gt_accuracy_matrix, 
                         index=[f'Rep_{i}' for i in range(n_reps)],
                         columns=[f'Rep_{i}' for i in range(n_reps)])
    gt_df.to_csv(f'{base_path}/gt_accuracy_matrix_raw.csv')
    
    neuron_df = pd.DataFrame(neuron_accuracy_matrix,
                             index=[f'Rep_{i}' for i in range(n_reps)],
                             columns=[f'Rep_{i}' for i in range(n_reps)])
    neuron_df.to_csv(f'{base_path}/neuron_accuracy_matrix_raw.csv')

print(f"PDF 已保存至: {output_pdf}")
print(f"gt_accuracy_matrix.csv 已保存至: {base_path}/")
print(f"neuron_accuracy_matrix.csv 已保存至: {base_path}/")

print("\n" + "=" * 60)
print("分析完成!")
print("=" * 60)

步骤1: 读取 mouse2_0 到 mouse2_4 的数据
Rep 0: 66 neurons, 673013 spikes
Rep 1: 69 neurons, 709892 spikes
Rep 2: 68 neurons, 589107 spikes
Rep 3: 72 neurons, 714423 spikes
Rep 4: 74 neurons, 725828 spikes

共加载 5 个重复的数据

步骤2: 计算 gt_detect_array 5x5 匹配准确率矩阵
  Rep 0 vs Rep 0: 1.0000
  Rep 0 vs Rep 1: 0.7803
  Rep 0 vs Rep 2: 0.9226
  Rep 0 vs Rep 3: 0.8280
  Rep 0 vs Rep 4: 0.8918
  Rep 1 vs Rep 0: 0.8231
  Rep 1 vs Rep 1: 1.0000
  Rep 1 vs Rep 2: 0.8640
  Rep 1 vs Rep 3: 0.8942
  Rep 1 vs Rep 4: 0.8429
  Rep 2 vs Rep 0: 0.8076
  Rep 2 vs Rep 1: 0.7170
  Rep 2 vs Rep 2: 1.0000
  Rep 2 vs Rep 3: 0.7801
  Rep 2 vs Rep 4: 0.7588
  Rep 3 vs Rep 0: 0.8790
  Rep 3 vs Rep 1: 0.8999
  Rep 3 vs Rep 2: 0.9461
  Rep 3 vs Rep 3: 1.0000
  Rep 3 vs Rep 4: 0.8792
  Rep 4 vs Rep 0: 0.9618
  Rep 4 vs Rep 1: 0.8618
  Rep 4 vs Rep 2: 0.9349
  Rep 4 vs Rep 3: 0.8933
  Rep 4 vs Rep 4: 1.0000

gt_detect_array 匹配准确率矩阵:
[[1.         0.78029334 0.92264054 0.82802205 0.89183939]
 [0.82305097 1.         0.86399075 0.894191